
# 04 — Análise das Idades Críticas — UTKFace

## Objetivo

Este notebook realiza uma análise específica do desempenho dos modelos DeepFace e MiVOLO nas idades de referência de 14, 16 e 18 anos, consideradas relevantes para a avaliação dos mecanismos de estimativa de idade.

A análise utiliza os resultados obtidos nos experimentos com a base UTKFace, organizados em três partições.

## Métricas utilizadas

São consideradas duas métricas:

- **Erro Médio Absoluto (MAE):** representa a média dos erros absolutos entre as idades de referência e as idades estimadas.
- **Acurácia com tolerância de ±1 ano:** representa a proporção de estimativas cuja diferença em relação à idade de referência é de, no máximo, um ano.

Os resultados são apresentados separadamente por tecnologia, partição e idade de referência.


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


In [ ]:

import os
import pandas as pd
import numpy as np

# ============================================================
# CAMINHOS DOS RESULTADOS
# ============================================================

PASTA_RESULTADOS = (
    "/content/drive/MyDrive/Classroom/"
    "9923 31 - TRABALHO DE CONCLUSÃO DE CURSO 2026/"
    "DESENVOLVIMENTO/Ambientes Colab/resultados"
)

PASTA_RESULTADOS_DEEPFACE = os.path.join(
    PASTA_RESULTADOS,
    "DeepFace",
    "csv"
)

PASTA_RESULTADOS_MIVOLO = os.path.join(
    PASTA_RESULTADOS,
    "MiVOLO",
    "arquivos csv"
)

PASTA_SAIDA = PASTA_RESULTADOS

# ============================================================
# CONFIGURAÇÕES DA ANÁLISE
# ============================================================

IDADES_ANALISADAS = [14, 16, 18]

# Critério provisório: confirmar a tolerância com o orientador.
TOLERANCIA = 1

# ============================================================
# ARQUIVOS OFICIAIS
# ============================================================

arquivos_resultados = {
    "DeepFace": {
        "Parte 1": os.path.join(
            PASTA_RESULTADOS_DEEPFACE,
            "resultados_deepface_utkface_part1.csv"
        ),
        "Parte 2": os.path.join(
            PASTA_RESULTADOS_DEEPFACE,
            "resultados_deepface_utkface_part2.csv"
        ),
        "Parte 3": os.path.join(
            PASTA_RESULTADOS_DEEPFACE,
            "resultados_deepface_utkface_part3.csv"
        )
    },

    "MiVOLO": {
        "Parte 1": os.path.join(
            PASTA_RESULTADOS_MIVOLO,
            "resultados_mivolo_utkface_part1.csv"
        ),
        "Parte 2": os.path.join(
            PASTA_RESULTADOS_MIVOLO,
            "resultados_mivolo_utkface_part2.csv"
        ),
        "Parte 3": os.path.join(
            PASTA_RESULTADOS_MIVOLO,
            "resultados_mivolo_utkface_part3.csv"
        )
    }
}

print("Configurações concluídas.")
print(f"Idades analisadas: {IDADES_ANALISADAS}")
print(f"Tolerância configurada: ±{TOLERANCIA} ano(s)")


In [ ]:

# ============================================================
# ANÁLISE DAS IDADES CRÍTICAS
# ============================================================

resultados = []

for tecnologia, partes in arquivos_resultados.items():

    print("\n" + "=" * 65)
    print(f"TECNOLOGIA: {tecnologia}")
    print("=" * 65)

    for parte, caminho in partes.items():

        print(f"\nProcessando {parte}...")

        # Verifica a existência do arquivo
        if not os.path.isfile(caminho):
            raise FileNotFoundError(
                f"Arquivo não encontrado: {caminho}"
            )

        # Carrega os resultados
        df = pd.read_csv(caminho)

        # Verifica as colunas necessárias
        colunas_necessarias = [
            "Idade Real",
            "Idade Estimada",
            "Erro Absoluto",
            "Status"
        ]

        for coluna in colunas_necessarias:
            if coluna not in df.columns:
                raise ValueError(
                    f"Coluna '{coluna}' não encontrada em {caminho}"
                )

        # ----------------------------------------------------
        # FILTRAGEM DAS INFERÊNCIAS BEM-SUCEDIDAS
        # ----------------------------------------------------

        status_valido = (
            "Sucesso" if tecnologia == "DeepFace" else "OK"
        )

        df_validos = df[
            df["Status"].astype(str).str.strip() == status_valido
        ].copy()

        # Conversão das colunas para valores numéricos
        colunas_numericas = [
            "Idade Real",
            "Idade Estimada",
            "Erro Absoluto"
        ]

        for coluna in colunas_numericas:
            df_validos[coluna] = pd.to_numeric(
                df_validos[coluna],
                errors="coerce"
            )

        # Mantém somente registros com valores numéricos
        # finitos nas três colunas utilizadas na análise.
        df_validos = df_validos[
            np.isfinite(df_validos["Idade Real"])
            & np.isfinite(df_validos["Idade Estimada"])
            & np.isfinite(df_validos["Erro Absoluto"])
        ].copy()

        # ----------------------------------------------------
        # ANÁLISE INDIVIDUAL DAS IDADES CRÍTICAS
        # ----------------------------------------------------

        for idade in IDADES_ANALISADAS:

            df_idade = df_validos[
                df_validos["Idade Real"] == idade
            ].copy()

            if df_idade.empty:
                print(
                    f"{parte} | Idade {idade}: "
                    "Nenhum registro válido encontrado."
                )
                continue

            # ------------------------------------------------
            # CÁLCULO DO MAE
            # ------------------------------------------------

            mae = df_idade["Erro Absoluto"].mean()

            # ------------------------------------------------
            # CLASSIFICAÇÃO PELA TOLERÂNCIA
            # ------------------------------------------------

            df_idade["Acerto Tolerancia"] = (
                (
                    df_idade["Idade Estimada"]
                    - df_idade["Idade Real"]
                ).abs() <= TOLERANCIA
            )

            acertos = int(
                df_idade["Acerto Tolerancia"].sum()
            )

            total = len(df_idade)

            erros = total - acertos

            acuracia_tolerancia = (
                acertos / total
            ) * 100

            # ------------------------------------------------
            # ARMAZENAMENTO DOS RESULTADOS
            # ------------------------------------------------

            resultados.append({
                "Tecnologia": tecnologia,
                "Base": "UTKFace",
                "Parte": parte,
                "Idade Real": idade,
                "Quantidade": total,
                "MAE (anos)": mae,
                f"Acertos ±{TOLERANCIA} ano(s)": acertos,
                f"Erros ±{TOLERANCIA} ano(s)": erros,
                f"Acurácia ±{TOLERANCIA} ano(s) (%)":
                    acuracia_tolerancia
            })

            print(
                f"{parte} | "
                f"Idade {idade} | "
                f"N = {total} | "
                f"MAE = {mae:.2f} | "
                f"Acurácia ±{TOLERANCIA} = "
                f"{acuracia_tolerancia:.2f}%"
            )

print("\nAnálise concluída.")


In [ ]:

# ============================================================
# ORGANIZAÇÃO DA TABELA FINAL
# ============================================================

df_resultados_limiares = pd.DataFrame(resultados)

if df_resultados_limiares.empty:
    raise ValueError(
        "Nenhum resultado válido foi encontrado para a análise."
    )

# Arredondamento dos resultados
colunas_percentuais = [
    coluna for coluna in df_resultados_limiares.columns
    if coluna.startswith("Acurácia")
]

df_resultados_limiares["MAE (anos)"] = (
    df_resultados_limiares["MAE (anos)"].round(2)
)

for coluna in colunas_percentuais:
    df_resultados_limiares[coluna] = (
        df_resultados_limiares[coluna].round(2)
    )

# ============================================================
# ORDENAÇÃO DOS RESULTADOS
# ============================================================

ordem_tecnologia = [
    "DeepFace",
    "MiVOLO"
]

ordem_parte = [
    "Parte 1",
    "Parte 2",
    "Parte 3"
]

df_resultados_limiares["Tecnologia"] = pd.Categorical(
    df_resultados_limiares["Tecnologia"],
    categories=ordem_tecnologia,
    ordered=True
)

df_resultados_limiares["Parte"] = pd.Categorical(
    df_resultados_limiares["Parte"],
    categories=ordem_parte,
    ordered=True
)

df_resultados_limiares = (
    df_resultados_limiares
    .sort_values(
        [
            "Tecnologia",
            "Parte",
            "Idade Real"
        ]
    )
    .reset_index(drop=True)
)

# ============================================================
# EXIBIÇÃO DOS RESULTADOS
# ============================================================

print("=" * 80)
print("RESULTADOS DAS IDADES CRÍTICAS — UTKFace")
print("=" * 80)

display(df_resultados_limiares)

# ============================================================
# EXPORTAÇÃO DO CSV
# ============================================================

arquivo_saida = os.path.join(
    PASTA_SAIDA,
    "resultados_idades_criticas_utkface.csv"
)

df_resultados_limiares.to_csv(
    arquivo_saida,
    index=False,
    encoding="utf-8-sig"
)

print("\nTabela exportada com sucesso!")
print(f"Arquivo: {arquivo_saida}")
